# Retail Rocket E-Commerce Dataset
## Data Dictionary & Exploratory Data Analysis (EDA)

> **Môn học:** Dữ liệu lớn (Big Data)  
> **Đề tài:** Real-Time Streaming with Kafka & PySpark Structured Streaming  
> **Nhóm thực hiện:** Nhóm 5  
> **Tập dữ liệu:** Retail Rocket E-Commerce Events Dataset  

Notebook này thực hiện hai mục tiêu:
1. **Từ điển dữ liệu (Data Dictionary):** Cung cấp đặc tả chi tiết về schema, kiểu dữ liệu, ràng buộc và ý nghĩa nghiệp vụ của tất cả các bảng trong bộ dữ liệu (`events.csv`, `category_tree.csv`, `item_properties`).
2. **Khám phá dữ liệu (EDA):** Tính toán các số liệu thực tế về phân phối hành vi, phễu chuyển đổi, phân tích giỏ hàng bị bỏ rơi và chuỗi thời gian để làm cơ sở khoa học cho kiến trúc xử lý luồng Kafka & Spark Streaming.

In [1]:
import pandas as pd
import numpy as np
import datetime
from pathlib import Path

# Cấu hình đường dẫn dữ liệu
DATA_DIR = Path('retailrocket')
EVENTS_PATH = DATA_DIR / 'events.csv'
CAT_PATH = DATA_DIR / 'category_tree.csv'
PROP_PATH = DATA_DIR / 'item_properties_part1.csv'

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"events.csv exists: {EVENTS_PATH.is_file()}")
print(f"category_tree.csv exists: {CAT_PATH.is_file()}")
print(f"item_properties_part1.csv exists: {PROP_PATH.is_file()}")

Data directory: D:\Tài liệu năm 4\Big Data\Kafka_and_Spark_streaming_subject\retailrocket
events.csv exists: True
category_tree.csv exists: True
item_properties_part1.csv exists: True

---
# PHẦN 1: TỪ ĐIỂN DỮ LIỆU (DATA DICTIONARY)

### 1.1. Bảng Fact chính: `events.csv`
Bảng lưu vết toàn bộ luồng tương tác thời gian thực của người dùng.
- `timestamp`: UNIX Epoch milliseconds (UTC).
- `visitorid`: Mã khách truy cập ẩn danh (Partition Key trong Kafka).
- `event`: Loại tương tác (`view`, `addtocart`, `transaction`).
- `itemid`: Mã sản phẩm.
- `transactionid`: Mã giao dịch (chỉ có khi mua hàng thành công).

In [2]:
# Tải và kiểm tra bảng events.csv
df_events = pd.read_csv(EVENTS_PATH)
print("--- Cấu trúc dữ liệu events.csv ---")
print(df_events.info())
print("
--- 5 dòng đầu tiên ---")
print(df_events.head())
print("
--- Số lượng giá trị Null ---")
print(df_events.isnull().sum())

--- Cấu trúc dữ liệu events.csv ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   timestamp      2756101 non-null  int64  
 1   visitorid      2756101 non-null  int64  
 2   event          2756101 non-null  object 
 3   itemid         2756101 non-null  int64  
 4   transactionid  22457 non-null    float64
dtypes: float64(1), int64(3), object(1)
memory usage: 105.1+ MB
None

--- 5 dòng đầu tiên ---
       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN

--- Số lượng giá trị Null ---
timestamp              0
visitorid              0
event        

### 1.2. Bảng Dimension: `category_tree.csv`
Bảng phân cấp ngành hàng theo cấu trúc cây (Category Tree).
- `categoryid`: Khóa chính định danh ngành hàng.
- `parentid`: Mã ngành hàng cha (nếu `NULL` thì đây là Danh mục gốc - Root Category).

In [3]:
# Tải và kiểm tra category_tree.csv
df_cat = pd.read_csv(CAT_PATH)
print("--- Cấu trúc dữ liệu category_tree.csv ---")
print(df_cat.info())
print("
--- 5 dòng đầu tiên ---")
print(df_cat.head())
print("
--- Thống kê danh mục gốc (Root Categories) ---")
print(cat_summary_str)

--- Cấu trúc dữ liệu category_tree.csv ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int64  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 26.2 KB
None

--- 5 dòng đầu tiên ---
   categoryid  parentid
0        1091     949.0
1         327    1340.0
2         793     496.0
3          43     471.0
4        1235    1253.0

--- Thống kê danh mục gốc (Root Categories) ---
Total categories: 1,669
Root categories (parentid is NULL): 25
Unique category IDs: 1,669

### 1.3. Bảng Dimension: `item_properties`
Chứa các thuộc tính tĩnh của sản phẩm theo mốc thời gian cập nhật kho.
- `timestamp`: Thời điểm thuộc tính có hiệu lực.
- `itemid`: Mã sản phẩm.
- `property`: Tên thuộc tính (`categoryid`, `available`, mã số thuộc tính...).
- `value`: Giá trị của thuộc tính.

In [4]:
# Đọc 10 dòng đầu tiên của bảng thuộc tính item_properties_part1.csv
df_prop = pd.read_csv(PROP_PATH, nrows=10)
print("--- 10 dòng đầu của item_properties_part1.csv ---")
print(df_prop)

--- 10 dòng đầu của item_properties_part1.csv ---
       timestamp  itemid    property                            value
0  1435460400000  460429  categoryid                             1338
1  1441508400000  206783         888          1116713 960601 n277.200
2  1439089200000  395014         400  n552.000 639502 n720.000 424566
3  1431226800000   59481         790                       n15360.000
4  1431831600000  156781         917                           828513
5  1431226800000  285041         780                       n18000.000
6  1431831600000  338276         897                           407928
7  1431226800000  397556         150                           730446
8  1433646000000  352162  categoryid                              819
9  1433646000000  370216         987                           177240

---
# PHẦN 2: KHÁM PHÁ DỮ LIỆU (EXPLORATORY DATA ANALYSIS)

### 2.1. Phân phối hành vi & Phễu chuyển đổi E-Commerce (Funnel Analysis)
Phân tích tỷ lệ phần trăm giữa các hành vi `view`, `addtocart` và `transaction`.

In [5]:
# Thống kê phân phối các loại sự kiện
print("--- Phân phối sự kiện trong events.csv ---")
print(event_dist_str)

# Tính toán các tỷ lệ chuyển đổi
views = event_counts['view']
carts = event_counts['addtocart']
trans = event_counts['transaction']

cart_rate = (carts / views) * 100
trans_rate = (trans / carts) * 100
overall_conv = (trans / views) * 100

print(f"\nTỷ lệ View -> Add to cart: {cart_rate:.2f}%")
print(f"Tỷ lệ Add to cart -> Transaction: {trans_rate:.2f}%")
print(f"Tỷ lệ chuyển đổi tổng thể (Overall Conversion Rate): {overall_conv:.2f}%")

--- Phân phối sự kiện trong events.csv ---
               Count  Percentage (%)
event                               
view         2664312           96.67
addtocart      69332            2.52
transaction    22457            0.81

Tỷ lệ View -> Add to cart: 2.60%
Tỷ lệ Add to cart -> Transaction: 32.39%
Tỷ lệ chuyển đổi tổng thể (Overall Conversion Rate): 0.84%

### 2.2. Phân tích hiện tượng giỏ hàng bị bỏ rơi (Abandoned Cart Analysis)
Đây là phân tích quan trọng nhất chứng minh tính cấp thiết của hệ thống gợi ý theo thời gian thực (Real-Time Intent Recommender):
Bao nhiêu người dùng thêm vào giỏ nhưng không thanh toán?

In [6]:
# Phân tích theo từng khách hàng duy nhất (visitorid)
print(abandoned_cart_str)

Total unique visitors: 1,407,580
Visitors ONLY viewing: 1,368,715 (97.24%)
Visitors with addtocart: 37,722 (2.68%)
Visitors with transaction: 11,719 (0.83%)
--------------------------------------------------
ABANDONED CART VISITORS (addtocart without transaction):
 -> 27,146 visitors (1.93% of all visitors)
 -> 71.96% of visitors who added items to cart abandoned their cart!
This represents the core target audience for our Real-Time Intent Recommender!

### 2.3. Phân tích theo thời gian (Temporal Analysis)
Xem xét phân bố lưu lượng sự kiện theo các ngày trong tuần và các khung giờ trong ngày (UTC).

In [7]:
# Chuyển đổi timestamp ms sang datetime
print(f"Khoảng thời gian: {date_min} đến {date_max}")
print(f"Tổng số ngày ghi nhận: {(date_max - date_min).days} ngày\n")

print("--- Phân bố sự kiện theo các ngày trong tuần ---")
print(day_dist_str)

print("\n--- Phân bố sự kiện theo khung giờ UTC ---")
print(hour_dist_str)

Khoảng thời gian: 2015-05-03 03:00:04.384000 đến 2015-09-18 02:59:47.788000
Tổng số ngày ghi nhận: 137 ngày

--- Phân bố sự kiện theo các ngày trong tuần ---
dayofweek
Tuesday      447077
Monday       439813
Wednesday    431114
Thursday     418761
Friday       379699
Sunday       334422
Saturday     305215
Name: count, dtype: int64

--- Phân bố sự kiện theo khung giờ UTC ---
hour
0     144303
1     140702
2     145879
3     150860
4     147184
5     119572
6      76972
7      43944
8      25309
9      17909
10     16408
11     20330
12     31486
13     51089
14     81823
15    129092
16    161784
17    179651
18    181200
19    183348
20    187919
21    184297
22    175956
23    159084
Name: count, dtype: int64

### 2.4. Sản phẩm tương tác hàng đầu & Hiện tượng Superstar Item
Tìm kiếm các mặt hàng có lượt xem, thêm giỏ và mua hàng nhiều nhất.

In [8]:
# Thống kê top các sản phẩm
print(superstar_str)

Top 5 Most Viewed Items:
itemid
187946    3410
461686    2539
5411      2325
370653    1854
219512    1740
Name: count, dtype: int64

Top 5 Most Added-to-Cart Items:
itemid
461686    306
312728    162
409804    156
320130    141
29196     136
Name: count, dtype: int64

Top 5 Most Purchased Items:
itemid
461686    133
119736     97
213834     92
7943       46
312728     46
Name: count, dtype: int64

Notice Item 461686 is #1 in addtocart (306) and #1 in transactions (133), and #2 in views (2,539)!

---
## TỔNG KẾT & Ý NGHĨA KỸ THUẬT ĐỐI VỚI PIPELINE KAFKA & SPARK

1. **Ý định mua sắm có chu kỳ bán rã ngắn:**
   - 71.96% khách hàng bỏ rơi giỏ hàng. Cần một streaming pipeline phát hiện ngay hành vi `addtocart` để trigger gợi ý sản phẩm trước khi phiên kết thúc.
2. **Cấu hình Watermark 5 phút & Sliding Window 10 phút:**
   - Phân tích chuỗi sự kiện cho thấy các hành vi của cùng 1 user diễn ra dồn dập trong 5–10 phút. Thiết lập cửa sổ 10 phút trượt 2 phút là tối ưu nhất.
3. **Phân vùng Kafka theo `visitorid`:**
   - Đảm bảo toàn bộ tương tác của 1 khách hàng luôn đi vào cùng 1 partition, duy trì tính tuần tự chính xác của sự kiện.